# Custom Sinks and ParquetReader

Demonstrates `SinkRegistry`, custom sink creation, `ParquetReader`, and pipeline composition.

| # | Topic |
|---|---|
| 1 | SinkRegistry — listing and creating built-in sinks |
| 2 | CsvSink — streaming CSV output |
| 3 | JsonlSink — JSON lines output |
| 4 | ParquetSink / ParquetPipeline — partitioned Parquet output |
| 5 | ParquetReader — reading partitioned parquet datasets |
| 6 | register_sink — custom sink implementation |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import tempfile
from dataclasses import dataclass

import pandas as pd

from boti_data import (
    CsvSink,
    CsvSinkConfig,
    JsonlSink,
    JsonlSinkConfig,
    ParquetMaterializationResult,
    ParquetPipeline,
    ParquetSink,
    SinkRegistry,
    SinkWriteResult,
    SinkPipeline,
    available_sinks,
    create_sink,
    register_sink,
)
from boti_data import ParquetReader


## 1. SinkRegistry — listing and creating built-in sinks

The registry maintains a map of registered sink types and can instantiate them by name.

In [ ]:
# List all registered sinks
print("Available sinks:", available_sinks())

# Create a sink by type name
sink = create_sink("csv", {"storage_path": tempfile.mktemp(suffix=".csv")})
print(f"Created sink: {type(sink).__name__} -> {sink.config.storage_path}")


## 2. CsvSink — streaming CSV output

Writes a DataFrame to CSV with configurable options.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    csv_path = Path(tmp) / "output.csv"
    sink = CsvSink({"storage_path": str(csv_path)})

    df = pd.DataFrame({"id": [1, 2, 3], "value": [10.5, 20.5, 30.5]})
    result = sink.write(df, write_index=False)

    print(f"Write result: {result}")
    print(f"Written CSV:\n{list(csv_path.iterdir())}")


## 3. JsonlSink — JSON lines output

Writes each row as a JSON object on a separate line.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    jsonl_path = Path(tmp) / "output.jsonl"
    sink = JsonlSink({"storage_path": str(jsonl_path)})

    df = pd.DataFrame({
        "id": [1, 2],
        "name": ["alice", "bob"],
        "metadata": [["admin", "user"], ["viewer"]],
    })
    result = sink.write(df)

    print(f"Write result: {result}")
    print(f"Written JSONL:\n{list(jsonl_path.iterdir())}")


## 4. ParquetSink / ParquetPipeline — partitioned Parquet output

Writes DataFrames as partitioned Parquet datasets with optional partitioning by column.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    pq_path = Path(tmp) / "parquet_output"
    sink = ParquetSink({"parquet_storage_path": str(pq_path)}, partition_on=None)

    df = pd.DataFrame({
        "category": ["A", "B", "A", "B"],
        "value": [1.0, 2.0, 3.0, 4.0],
        "date": pd.to_datetime(["2024-01-01", "2024-01-01", "2024-02-01", "2024-02-01"]),
    })
    result = sink.write(df)

    print(f"Write result: {result}")
    print(f"Output files: {list(pq_path.rglob('*.parquet'))}")


### ParquetPipeline — materialization pipeline

A `ParquetPipeline` provides multi-step materialization with intermediate schema validation.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    # First write a parquet dataset
    data_path = str(Path(tmp) / "data")
    sink = ParquetSink({"parquet_storage_path": data_path}, partition_on=None)
    df = pd.DataFrame({
        "category": ["A", "B", "A", "B"],
        "value": [1.0, 2.0, 3.0, 4.0],
    })
    sink.write(df)

    # Now use ParquetReader as the source for a pipeline
    pipeline = ParquetPipeline(
        source=reader,
        destination={"parquet_storage_path": str(Path(tmp) / "pipeline_output")},
        partition_on=None,
    )
    result: ParquetMaterializationResult = pipeline.materialize()
    print(f"Paths: {result.paths if hasattr(result, 'paths') else result.path}")
    print(f"Files: {list(Path(result.path).rglob('*.parquet'))}")


## 5. ParquetReader — reading partitioned parquet datasets

Reads partitioned parquet datasets back with schema inference.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    # First write a partitioned parquet dataset
    sink = ParquetSink({"parquet_storage_path": str(Path(tmp) / "data")}, partition_on=None)
    df = pd.DataFrame({"group": ["a", "a", "b"], "val": [1, 2, 3]})
    sink.write(df)

    # Read it back with ParquetReader
    # Read all data
    data = reader.load(return_type="pandas")
    print(f"Data:\n{data}")
    print()
    # Read with filters
    filtered = reader.load(filters={"group": "a"}, return_type="pandas")
    print(f"Filtered:\n{filtered}")


## 6. register_sink — custom sink implementation

Any callable conforming to the sink protocol can be registered and created by name.

In [ ]:
from dataclasses import dataclass

@dataclass
class HtmlSinkConfig:
    include_index: bool = False

def html_sink_factory(config) -> "HtmlSink":
    path = Path(config["path"]) if isinstance(config, dict) else config.path
    kwargs = {k: v for k, v in config.items() if k != "path"} if isinstance(config, dict) else {}
    return HtmlSink(path, config=HtmlSinkConfig(**kwargs))

class HtmlSink:
    """Custom sink that writes a DataFrame as an HTML table."""
    def __init__(self, path: Path, config: HtmlSinkConfig | None = None):
        self.path = path
        self.config = config or HtmlSinkConfig()

    def write(self, df: pd.DataFrame) -> SinkWriteResult:
        html = df.to_html(index=self.config.include_index)
        self.path.write_text(html, encoding="utf-8")
        return SinkWriteResult(path=str(self.path), files=(str(self.path),))
# Register the custom sink type
register_sink("html", html_sink_factory)
print("Registered sinks:", available_sinks())

# Use the custom sink
with tempfile.TemporaryDirectory() as tmp:
    sink = create_sink("html", {"path": str(Path(tmp) / "report.html")})
    df = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
    result = sink.write(df)
    print(f"Write result: {result}")
    report_path = Path(tmp) / "report.html"
    print(f"\nHTML content:\n{report_path.read_text()}")


### Summary

- **`SinkRegistry`** maps sink type names to factory functions.
- **`CsvSink`**, **`JsonlSink`**, **`ParquetSink`** — built-in sink implementations for common formats.
- **`ParquetPipeline`** — multi-step parquet materialization with partitioning.
- **`ParquetReader`** — reads partitioned parquet datasets with schema inference and filtering.
- **`register_sink`** / **`create_sink`** — extensible sink system; implement `write(df) -> SinkWriteResult`.